# Apertus StressLab: evidence and free Colab preparation

This notebook verifies the completed Phase 3B and eight Colab diagnostic archives without a model. **Run all performs no inference by default.** Python and Git suffice; use a CPU runtime for evidence review. Recorded diagnostics are published with their original hashes and failed attempts; review the linked reports before interpreting numerical functionality as semantic robustness.

Optional preparation cells below create the recorded inference environment and download the pinned model cache only if their flags are explicitly enabled. Select a **free T4 GPU** for environment preparation; no paid upgrade is required by this workflow. The setup recipe has been syntax-checked locally, but not executed in a new Colab runtime. If the base Python/PyTorch or fresh package inventory differs, stop and record it for a new prospective plan. No historical launcher, study suite, semantic evaluator or generation is run here.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

PROJECT_REF = "dbabc122ec8834bca3e274f821507b38bf760c18"
PREPARE_ENV = False
CACHE_MODEL = False
root = Path("/content/apertus-stresslab")
python = Path("/content/apertus-env/bin/python")

def run(args, **kwargs):
    return subprocess.run(args, check=True, **kwargs)

if not root.exists():
    run(["git", "clone", "https://github.com/Alextsvr/apertus-stresslab.git", str(root)])
assert not subprocess.check_output(
    ["git", "status", "--porcelain", "--untracked-files=no"], cwd=root, text=True
).strip(), "Tracked project files changed; preserve them before checkout"
run(["git", "fetch", "origin", "main"], cwd=root)
run(["git", "checkout", "--detach", PROJECT_REF], cwd=root)
print("Evidence checkpoint:", PROJECT_REF)


## 1. Verify published evidence, without GPU or token

The verifier reads stored bytes and labels. It never runs the model or evaluator. Expected: eight archives verified, original raw SHA256 `4/4 match`, original counts 22 PASS / 6 POTENTIAL / 2 DETECTED. The original controls remain six PASS.


In [ ]:
run([sys.executable, "scripts/verify_colab_evidence.py"], cwd=root)


## 2. Read the result and provenance

The isolated replay supports FP16 representation overflow at one projection coordinate. The FP32 intervention kept packed NF4 weights/state unchanged and completed short normal-generation checks plus one actual 96-token cached continuation. That covers specific technical prompts, not long contexts or general semantic robustness. The October 5 comparison reused already observed prompts; it is not new held-out evidence. Both preflight stops are preserved.


In [ ]:
from IPython.display import Markdown, display
catalog = json.loads((root / "evidence/colab/catalog.json").read_text())
rows = ["| Attempt | Stored outcome | Report |", "| --- | --- | --- |"]
for item in catalog["attempts"]:
    outcome = item["stored_result"].get("outcome", "completed configuration study")
    link = f"https://github.com/Alextsvr/apertus-stresslab/blob/{PROJECT_REF}/{item['report']}"
    rows.append(f"| {item['id']} | {outcome} | [Report]({link}) |")
display(Markdown("\n".join(rows)))
display(Markdown(f"[Full diagnostic overview](https://github.com/Alextsvr/apertus-stresslab/blob/{PROJECT_REF}/docs/colab_diagnostics/README.md)"))


## 3. Optional: prepare the recorded free T4 environment

Leave `PREPARE_ENV=False` for evidence review. To prepare for a separately planned future experiment, change that flag to true and select T4 in Runtime settings. This cell checks the base Python/PyTorch before installing, uses a venv without `ensurepip`, inherits Colab's existing PyTorch and avoids the inference requirement file's audio extras. It does not replace base PyTorch or run a forward pass. It installs into the venv, records a fresh preflight and never removes or overwrites an existing environment.

The environment must be empty on first preparation; preserve an existing environment rather than rerun this cell. A different inventory from the original probes is not automatically approved for their replay.


In [ ]:
if PREPARE_ENV:
    assert not python.parent.parent.exists(), "Environment exists; preserve it instead of repeating setup"
    run([sys.executable, "-c", "import sys,torch; assert sys.version_info[:3]==(3,13,15), 'Base Python changed'; assert torch.__version__=='2.11.0+cu130', 'Base PyTorch changed'"])
    run([sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", str(python.parent.parent)])
    pins = ["torch==2.11.0+cu130", "accelerate==1.15.0", "bitsandbytes==0.50.2",
            "tokenizers==0.22.2", "numpy==2.1.3", "huggingface_hub==1.31.0",
            "safetensors==0.8.0", "pydantic==2.13.5", "rich==13.9.4"]
    run([sys.executable, "-m", "pip", "--python", str(python), "install", *pins])
    run([sys.executable, "-m", "pip", "--python", str(python), "install", "--no-deps",
         "git+https://github.com/swiss-ai/transformers.git@3797303dda74844e3d1f8977ff5518bb91f818b4"])
    run([sys.executable, "-m", "pip", "--python", str(python), "install", "--no-deps", "-e", str(root)])
    run([str(python), "-c", "from transformers import AutoModelForMultimodalLM,AutoProcessor; print('Apertus imports: OK')"])
    before = subprocess.check_output([str(python), "scripts/colab_quantization.py", "--check-json"], cwd=root, text=True)
    preflight = json.loads(before)
    recorded = root / "results/colab-preparation-preflight.json"
    recorded.parent.mkdir(exist_ok=True)
    with recorded.open("x", encoding="utf-8") as target:
        json.dump(preflight, target, indent=2)
    print("PREPARATION_ONLY_OK — no model loaded, no inference or scoring")
else:
    print("Environment preparation disabled; evidence review only")


## 4. Optional: Hugging Face access and pinned model cache

Leave `CACHE_MODEL=False` unless you deliberately want to prepare the model for a separately planned experiment. This downloads about 17.13 GiB of weights but does not load the model or generate. Add your existing read token as Colab Secret **HF_TOKEN**, enable access for this notebook and ensure your account has access to the checkpoint. The token is not printed or written to files. The model cache is temporary and is not part of published evidence. Notebook/Secret access is separate from a previous notebook's access.


In [ ]:
if CACHE_MODEL:
    assert python.is_file(), "Prepare the environment first"
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ.pop("HF_HUB_OFFLINE", None)
    download = r"""
import json
from pathlib import Path
from huggingface_hub import hf_hub_download,snapshot_download
model='swiss-ai/Apertus-v1.5-8B'
revision='a411d838600baf0e3635a3daf66fb7c55fc97bb6'
index=json.loads(Path(hf_hub_download(model,'model.safetensors.index.json',revision=revision)).read_text())
shards=sorted(set(index['weight_map'].values()))
assert len(shards)==6
snapshot=Path(snapshot_download(repo_id=model,revision=revision,
    allow_patterns=shards+['*.json','*.txt','*.jinja','*.model'],max_workers=2))
assert all((snapshot/name).is_file() for name in shards)
Path('/content/apertus-pinned-snapshot.txt').write_text(str(snapshot))
print('CACHE_READY — 6/6 shards; no model loaded or inference')
"""
    run([str(python), "-u", "-c", download], cwd=root)
else:
    print("Model download disabled")


## 5. Finish or plan the next experiment

There is deliberately no generation cell. A new semantic experiment needs a separate prospective plan, prompts, decoding/precision settings, current environment record, output-quality gates and an exclusive output directory. Do not rerun dated historical launchers or rescore the original Phase 3B. Save the notebook and download any new setup metadata you want to retain before ending the runtime; the published evidence is already on GitHub.

For shutting down, use Runtime → Disconnect and delete runtime. The temporary environment/cache will be discarded. [Colab FAQ](https://research.google.com/colaboratory/faq.html).


## 6. Export the next-session preparation record

For the next session, follow [the prospective preparation plan](https://github.com/Alextsvr/apertus-stresslab/blob/main/docs/colab_next_session_preparation.md): set `PREPARE_ENV=True`, keep `CACHE_MODEL=False`, and run the notebook once in a fresh free T4 runtime. After successful preparation, this cell downloads a small preflight ZIP and its SHA256 sidecar. It reads the existing record without repeating checks or loading a model. Keep both files and send them for review before any inference protocol is finalized. If an earlier cell fails, stop and preserve its output instead of continuing or repeating setup.


In [ ]:
if PREPARE_ENV:
    import hashlib, zipfile
    from datetime import datetime, timezone
    from google.colab import files

    recorded = root / 'results/colab-preparation-preflight.json'
    content = recorded.read_bytes()
    preflight = json.loads(content)
    assert preflight['weights_loaded'] is False
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    archive = Path('/content') / f'colab-next-session-preflight-{stamp}.zip'
    manifest = f'{hashlib.sha256(content).hexdigest()}  preflight.json\n'
    with zipfile.ZipFile(archive, 'x', zipfile.ZIP_DEFLATED) as exported:
        exported.writestr('preflight.json', content)
        exported.writestr('SHA256SUMS', manifest)
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    sidecar = archive.with_suffix('.zip.sha256')
    with sidecar.open('x', encoding='utf-8', newline='\n') as target:
        target.write(f'{digest}  {archive.name}\n')
    print('PREPARATION_EXPORT_OK')
    print('SHA256:', digest)
    files.download(str(archive))
    files.download(str(sidecar))
else:
    print('Preflight export disabled; no preparation was requested')


In [1]:
from pathlib import Path
import subprocess

root = Path("/content/apertus-stresslab")
commit = "7f080ef693c0d15910b71b77f2efaf2944442508"

if not root.exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/Alextsvr/apertus-stresslab.git",
        str(root)
    ], check=True)

assert not subprocess.check_output(
    ["git", "status", "--porcelain", "--untracked-files=no"],
    cwd=root, text=True
).strip(), "Есть изменения файлов проекта — остановись"

subprocess.run(
    ["git", "fetch", "origin", "main"],
    cwd=root, check=True
)
subprocess.run(
    ["git", "checkout", "--detach", commit],
    cwd=root, check=True
)

helper = root / "scripts/colab_fp32_context_cell.py"
exec(
    compile(helper.read_text(), str(helper), "exec"),
    {"MODE": "prepare"}
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 53.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Not uninstalling huggingface-hub at /usr/local/lib/python3.13/dist-packages, outside environment /content/apertus-env
    Can't uninstall 'huggingface_hub'. No files were found to uninstall.
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.2
    Not uninstalling tokenizers at /usr/local/lib/python3.13/dist-packages, outside environment /content/apertus-env
    Can't uninstall 'tokenizers'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.18.0 requir

In [2]:
from pathlib import Path
import subprocess

root = Path("/content/apertus-stresslab")
assert subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=root, text=True
).strip() == "7f080ef693c0d15910b71b77f2efaf2944442508"

helper = root / "scripts/colab_fp32_context_cell.py"
exec(
    compile(helper.read_text(), str(helper), "exec"),
    {"MODE": "cache"}
)

Preflight OK; downloading pinned cache only

Fetching 15 files: 100%|██████████| 15/15 [03:04<00:00, 12.33s/it]
CACHE_READY: 6/6 shards; no model loaded or inference


In [3]:
from pathlib import Path
import subprocess

root = Path("/content/apertus-stresslab")
assert subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=root, text=True
).strip() == "7f080ef693c0d15910b71b77f2efaf2944442508"

helper = root / "scripts/colab_fp32_context_cell.py"
exec(
    compile(helper.read_text(), str(helper), "exec"),
    {"MODE": "run"}
)

Loading cached NF4 weights once; no forward before FP32 promotion
[transformers] CUDA-fused xIELU not available (No module named 'xielu') – falling back to a Python version.
For CUDA xIELU (experimental), `pip install git+https://github.com/nickjbrowning/XIELU`

Loading weights: 100%|██████████| 924/924 [01:15<00:00, 12.25it/s]
Generating once: CTX-004-LOW-B-C (1/108)
Generating once: CTX-004-LOW-D-C (2/108)
Generating once: CTX-004-LOW-F-C (3/108)
Generating once: CTX-006-LOW-B-C (4/108)
Generating once: CTX-006-LOW-F-C (5/108)
Generating once: CTX-006-LOW-D-C (6/108)
Generating once: CTX-006-HIGH-D-C (7/108)
Generating once: CTX-006-HIGH-B-C (8/108)
Generating once: CTX-006-HIGH-F-C (9/108)
Generating once: CTX-001-LOW-D-C (10/108)
Generating once: CTX-001-LOW-F-C (11/108)
Generating once: CTX-001-LOW-B-C (12/108)
Generating once: CTX-005-LOW-F-C (13/108)
Generating once: CTX-005-LOW-B-C (14/108)
Generating once: CTX-005-LOW-D-C (15/108)
Generating once: CTX-004-HIGH-F-C (16/108)
Gen

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
from pathlib import Path
import subprocess

root = Path("/content/apertus-stresslab")

if not root.exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/Alextsvr/apertus-stresslab.git",
        str(root),
    ], check=True)

assert not subprocess.check_output(
    ["git", "status", "--porcelain", "--untracked-files=no"],
    cwd=root, text=True,
).strip(), "Обнаружены изменения файлов проекта — остановись"

subprocess.run(
    ["git", "fetch", "origin", "main"],
    cwd=root, check=True,
)
subprocess.run([
    "git", "checkout", "--detach",
    "38bcdf0bd639070492899309baf9903476363a31",
], cwd=root, check=True)

helper = root / "scripts/colab_qwen_context_cell.py"
exec(
    compile(helper.read_text(), str(helper), "exec"),
    {"MODE": "prepare"},
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 47.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Not uninstalling huggingface-hub at /usr/local/lib/python3.13/dist-packages, outside environment /content/apertus-env
    Can't uninstall 'huggingface_hub'. No files were found to uninstall.
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.2
    Not uninstalling tokenizers at /usr/local/lib/python3.13/dist-packages, outside environment /content/apertus-env
    Can't uninstall 'tokenizers'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.18.0 requir

In [2]:
from pathlib import Path

helper = Path(
    "/content/apertus-stresslab/scripts/colab_qwen_context_cell.py"
)
exec(
    compile(helper.read_text(), str(helper), "exec"),
    {"MODE": "cache"},
)

Preflight OK; downloading pinned cache only

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 13 files: 100%|██████████| 13/13 [05:40<00:00, 26.16s/it]
CACHE_READY: 4/4 shards; no model loaded or inference


In [3]:
from pathlib import Path

helper = Path(
    "/content/apertus-stresslab/scripts/colab_qwen_context_cell.py"
)
exec(
    compile(helper.read_text(), str(helper), "exec"),
    {"MODE": "run"},
)

Loading cached NF4 weights once; no forward before FP32 promotion

Loading weights: 100%|██████████| 339/339 [01:04<00:00,  5.25it/s]
Generating once: CTX-004-LOW-B-C (1/108)
Generating once: CTX-004-LOW-D-C (2/108)
Generating once: CTX-004-LOW-F-C (3/108)
Generating once: CTX-006-LOW-B-C (4/108)
Generating once: CTX-006-LOW-F-C (5/108)
Generating once: CTX-006-LOW-D-C (6/108)
Generating once: CTX-006-HIGH-D-C (7/108)
Generating once: CTX-006-HIGH-B-C (8/108)
Generating once: CTX-006-HIGH-F-C (9/108)
Generating once: CTX-001-LOW-D-C (10/108)
Generating once: CTX-001-LOW-F-C (11/108)
Generating once: CTX-001-LOW-B-C (12/108)
Generating once: CTX-005-LOW-F-C (13/108)
Generating once: CTX-005-LOW-B-C (14/108)
Generating once: CTX-005-LOW-D-C (15/108)
Generating once: CTX-004-HIGH-F-C (16/108)
Generating once: CTX-004-HIGH-D-C (17/108)
Generating once: CTX-004-HIGH-B-C (18/108)
Generating once: CTX-002-LOW-B-C (19/108)
Generating once: CTX-002-LOW-D-C (20/108)
Generating once: CTX-002-LOW-

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>